# v7 Python Authoring Quickstart

This notebook shows the new Python authoring layer on top of the existing `v7` training pipeline.

How it works:
- Python owns the project spec: model dimensions, template choice, data source, and train config.
- `TrainingProject.materialize()` calls the existing `version/v7/scripts/ck_run_v7.py init ... --generate-ir --generate-runtime` path.
- `TrainingProject.train()` calls the existing `version/v7/scripts/ck_run_v7.py train ...` path.
- `TrainingProject.prepare_viewers()` refreshes `ir_report.html`, prepares run-local viewer artifacts, and regenerates `ir_hub.html`.
- `dataset_viewer.html` and `attention.json` are conditional: they appear when the run has dataset manifests and tokenizer/probe artifacts.
- The actual runtime remains generated C code plus `libtrain.so`; Python is the authoring and orchestration layer.

How to start it from the repo root:
- `jupyter lab notebooks/v7_python_authoring_quickstart.ipynb`
- or `jupyter notebook notebooks/`

Open the notebook from inside the repo checkout so repo-root detection works.


In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

REPO_ROOT


In [ ]:
from pathlib import Path
import importlib.util
import sys
from IPython.display import HTML, display

REPO_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "ckernel_engine").exists() and (candidate / "version" / "v7").exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise RuntimeError(
        f"Could not find the C-Kernel-Engine repo root from cwd={Path.cwd()}. "
        "Open this notebook from inside the repo checkout."
    )

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

spec = importlib.util.find_spec("ckernel_engine")
if spec is None:
    raise RuntimeError(f"Repo root was found at {REPO_ROOT}, but Python still cannot import ckernel_engine")

import importlib
import ckernel_engine.v7.authoring as _authoring
import ckernel_engine.v7 as _v7
importlib.reload(_authoring)
importlib.reload(_v7)

from ckernel_engine.v7 import (
    DataSource,
    MaterializeOptions,
    TemplateSpec,
    TinyModelSpec,
    TokenizerPlan,
    TrainConfig,
    TrainingProject,
)

print(f"Imported ckernel_engine.v7 from repo root: {REPO_ROOT}")


In [ ]:
project = TrainingProject(
    run_name="python-ui-notebook-demo",
    model=TinyModelSpec(
        init="xavier_uniform",
        layers=2,
        vocab_size=256,
        embed_dim=128,
        hidden_dim=256,
        num_heads=8,
        num_kv_heads=4,
        context_len=128,
    ),
    template=TemplateSpec.builtin_template("qwen3"),
    tokenizer=TokenizerPlan(
        family="runtime_default",
        notes="Notebook keeps tokenizer ownership in the existing v7 runtime path.",
    ),
)

project


In [ ]:
materialize_result = project.materialize(
    MaterializeOptions(
        generate_ir=True,
        generate_runtime=True,
        strict=True,
    )
)

materialize_result


In [ ]:
train_result = project.train(
    DataSource.inline_text(
        "C-Kernel-Engine notebook quickstart.",
        description="small inline training text",
    ),
    TrainConfig(
        backend="ck",
        strict=True,
        epochs=1,
        seq_len=8,
        total_tokens=64,
        grad_accum=2,
        lr=5e-4,
        parity_regimen="suggest",
        memory_check=False,
    ),
)

train_result


In [ ]:
viewer_artifacts = project.prepare_viewers()
viewer_artifacts


## What To Inspect

After the cells above run, the important artifacts are:
- `python_authoring_plan.json`: Python-side project spec and command history
- `weights_manifest.json`: manifest-first training state and embedded template contract
- `ir1_train_forward.json`: train forward IR
- `ir2_train_backward.json`: backward IR
- `layout_train.json`: planned memory layout
- `train_e2e_latest.json`: latest train report exported by the existing `v7` runner
- `ir_report.html`: generated IR visualizer for this run
- `embeddings.json`: exported token embedding matrix when viewer prep runs against training weights
- `attention.json`: attention export when tokenizer/probe artifacts are available
- `dataset_viewer.html`: run-local dataset/token artifact viewer when dataset manifests are available
- `ir_hub.html`: shared run hub under the parent models root
- `generated_train_runtime_v7.c`: generated training runtime source
- `libtrain.so`: compiled generated runtime


In [ ]:
run_dir = train_result.run_dir
display(HTML(project.notebook_artifact_dashboard_html(title="Quickstart Artifact Dashboard")))
sorted(p.name for p in run_dir.iterdir())
